# Memahami & Membersihkan Data Sephora Skincare

**Di notebook ini:**
1. Memuat data produk dan ulasan
2. Mengecek kualitas data (kolom kosong, duplikat, tipe data)
3. Membersihkan data & membuat kolom baru
4. Menyimpan data bersih ke `data/processed/` untuk langkah berikutnya

In [4]:
import glob
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

from pathlib import Path

# Folder proyek (bekerja baik dijalankan dari root maupun dari folder notebooks/)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

In [5]:
products = pd.read_csv(f"{RAW}/product_info.csv")
print("Ukuran data produk:", products.shape)
products.head()

Ukuran data produk: (8494, 27)


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,variation_desc,ingredients,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.64,11.00,NaN,NaN,NaN,NaN,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD A...",35.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.15,13.00,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.00,30.00
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.25,16.00,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.00,30.00
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.48,21.00,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.00,30.00
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.23,13.00,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.00,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.00,30.00


In [7]:
review_files = sorted(glob.glob(f"{RAW}/reviews_*.csv"))
print("File ulasan:", review_files)

reviews = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in review_files],
    ignore_index=True,
)
print("Ukuran data ulasan:", reviews.shape)
reviews.head()

File ulasan: ['c:\\Users\\Nakata\\Documents\\Data Analyst\\sephora-skincare-analysis\\data\\raw\\reviews_0-250.csv', 'c:\\Users\\Nakata\\Documents\\Data Analyst\\sephora-skincare-analysis\\data\\raw\\reviews_1250-end.csv', 'c:\\Users\\Nakata\\Documents\\Data Analyst\\sephora-skincare-analysis\\data\\raw\\reviews_250-500.csv', 'c:\\Users\\Nakata\\Documents\\Data Analyst\\sephora-skincare-analysis\\data\\raw\\reviews_500-750.csv', 'c:\\Users\\Nakata\\Documents\\Data Analyst\\sephora-skincare-analysis\\data\\raw\\reviews_750-1250.csv']
Ukuran data ulasan: (1094411, 19)


,Unnamed: 0,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,0,1741593524,5,1.00,1.00,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.00
1,1,31423088263,1,0.00,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.00
2,2,5061282401,5,1.00,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.00
3,3,6083038851,5,1.00,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.00
4,4,47056667835,5,1.00,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.00


Pertanyaan yang kita jawab:
- Kolom mana yang banyak kosong?
- Apakah ada data duplikat?
- Apakah semua ulasan punya pasangan produknya?

In [15]:
def missing_report(df):
    pct = df.isna().mean().mul(100).round(1)
    return pct[pct > 0].sort_values(ascending=False).to_frame("% kosong")

print("=== Produk ===")
display(missing_report(products))
print("=== Ulasan ===")
display(missing_report(reviews))

=== Produk ===


,% kosong
sale_price_usd,96.80
value_price_usd,94.70
variation_desc,85.30
child_max_price,67.60
child_min_price,67.60
highlights,26.00
size,19.20
variation_value,18.80
variation_type,17.00
tertiary_category,11.70


=== Ulasan ===


,% kosong
helpfulness,51.30
review_title,28.40
hair_color,20.70
eye_color,19.20
skin_tone,15.60
is_recommended,15.30
skin_type,10.20
review_text,0.10


In [5]:
print("Duplikat product_id :", products["product_id"].duplicated().sum())
print("Duplikat baris ulasan:", reviews.duplicated().sum())
print("Ulasan tanpa produk  :", (~reviews["product_id"].isin(products["product_id"])).sum())

Duplikat product_id : 0


Duplikat baris ulasan: 0
Ulasan tanpa produk  : 0


### Ulasan hanya untuk produk Skincare
Data produk punya banyak kategori (Makeup, Hair, Fragrance, ...), tapi data ulasan **hanya** berisi produk Skincare. Karena itu proyek ini fokus ke **skincare**.

In [16]:
print("Kategori di data produk:")
display(products["primary_category"].value_counts())

reviewed = products[products["product_id"].isin(reviews["product_id"])]
print("Kategori produk yang punya ulasan:")
reviewed["primary_category"].value_counts()

Kategori di data produk:


primary_category
Skincare           2420
Makeup             2369
Hair               1464
Fragrance          1432
Bath & Body         405
Mini Size           288
Men                  60
Tools & Brushes      52
Gifts                 4
Name: count, dtype: int64

Kategori produk yang punya ulasan:


primary_category
Skincare    2351
Name: count, dtype: int64

## Membersihkan data produk
Keputusan pembersihan:
- Ambil hanya produk **Skincare**
- Buang kolom yang >85% kosong (`sale_price_usd`, `value_price_usd`, `variation_desc`) karena tidak bisa dianalisis dengan andal
- Buat kolom **`price_band`** (kelompok harga) supaya mudah dibandingkan

In [17]:
skincare = products[products["primary_category"] == "Skincare"].copy()
skincare = skincare.drop(columns=["sale_price_usd", "value_price_usd", "variation_desc"])

price_bins = [0, 25, 50, 100, np.inf]
price_labels = ["< $25", "$25-50", "$50-100", "> $100"]
skincare["price_band"] = pd.cut(skincare["price_usd"], bins=price_bins, labels=price_labels, right=False)

print("Jumlah produk skincare:", len(skincare))
skincare["price_band"].value_counts().sort_index()

Jumlah produk skincare: 2420


price_band
< $25      485
$25-50     921
$50-100    746
> $100     268
Name: count, dtype: int64

## Membersihkan data ulasan
Keputusan pembersihan:
- Buang kolom `Unnamed: 0` (hanya nomor indeks sisa ekspor)
- Ubah `submission_time` menjadi tipe tanggal, lalu ambil `year`
- Buang ulasan tanpa teks (sangat sedikit, ~0,1%)
- Tambah `review_length` (jumlah kata) dan `sentiment_group` berdasarkan rating
- Isi `skin_type` yang kosong dengan `"unknown"` (jangan dibuang, karena 10% data)

In [12]:
reviews_clean = reviews.drop(columns=["Unnamed: 0"]).copy()
reviews_clean["submission_time"] = pd.to_datetime(reviews_clean["submission_time"])
reviews_clean["year"] = reviews_clean["submission_time"].dt.year

before = len(reviews_clean)
reviews_clean = reviews_clean.dropna(subset=["review_text"])
print(f"Ulasan tanpa teks dibuang: {before - len(reviews_clean):,}")

reviews_clean["review_length"] = reviews_clean["review_text"].str.split().str.len()
reviews_clean["sentiment_group"] = pd.cut(
    reviews_clean["rating"], bins=[0, 2, 3, 5], labels=["negatif (1-2)", "netral (3)", "positif (4-5)"]
)
for col in ["skin_type", "skin_tone"]:
    reviews_clean[col] = reviews_clean[col].fillna("unknown")

# Tambahkan info kategori & kelompok harga dari tabel produk
reviews_clean = reviews_clean.merge(
    skincare[["product_id", "secondary_category", "tertiary_category", "price_band"]],
    on="product_id", how="left",
)
print("Ukuran ulasan bersih:", reviews_clean.shape)
reviews_clean.head()

Ulasan tanpa teks dibuang: 1,444
Ukuran ulasan bersih: (1092967, 24)


,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,year,review_length,sentiment_group,secondary_category,tertiary_category,price_band
0,1741593524,5,1.00,1.00,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,unknown,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.00,2023,79,positif (4-5),Cleansers,NaN,< $25
1,31423088263,1,0.00,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,unknown,NaN,unknown,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.00,2023,28,negatif (1-2),Lip Balms & Treatments,NaN,< $25
2,5061282401,5,1.00,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.00,2023,53,positif (4-5),Lip Balms & Treatments,NaN,< $25
3,6083038851,5,1.00,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,Can't go wrong with any of them,unknown,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.00,2023,45,positif (4-5),Lip Balms & Treatments,NaN,< $25
4,47056667835,5,1.00,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.00,2023,46,positif (4-5),Lip Balms & Treatments,NaN,< $25


## Ringkasan cepat data bersih

In [13]:
summary = {
    "Produk skincare": f"{len(skincare):,}",
    "Brand": f"{skincare['brand_name'].nunique():,}",
    "Ulasan": f"{len(reviews_clean):,}",
    "Pengulas unik": f"{reviews_clean['author_id'].nunique():,}",
    "Periode ulasan": f"{reviews_clean['submission_time'].min():%Y-%m} s/d {reviews_clean['submission_time'].max():%Y-%m}",
    "Rata-rata rating ulasan": f"{reviews_clean['rating'].mean():.2f}",
    "% ulasan positif (4-5)": f"{(reviews_clean['rating'] >= 4).mean():.1%}",
}
pd.Series(summary, name="Nilai").to_frame()

,Nilai
Produk skincare,"2,420"
Brand,143
Ulasan,"1,092,967"
Pengulas unik,"502,564"
Periode ulasan,2008-08 s/d 2023-03
Rata-rata rating ulasan,4.30
% ulasan positif (4-5),82.1%


In [10]:
reviews_clean["sentiment_group"].value_counts(normalize=True).mul(100).round(1).to_frame("% ulasan")

,% ulasan
sentiment_group,
positif (4-5),82.10
negatif (1-2),10.40
netral (3),7.50


## Simpan data bersih
Disimpan sebagai **Parquet** (lebih kecil & cepat dibaca dibanding CSV) untuk dipakai di notebook berikutnya.

In [14]:
skincare.to_parquet(f"{PROCESSED}/products_skincare.parquet", index=False)
reviews_clean.to_parquet(f"{PROCESSED}/reviews_clean.parquet", index=False)
print("Tersimpan di", PROCESSED)

Tersimpan di c:\Users\Nakata\Documents\Data Analyst\sephora-skincare-analysis\data\processed
